# Experiments

A workspace for your own experiments on the gendered-rules data. Run the two setup cells first, then change any example or use the empty cells at the bottom.

For an explanation of the data and method, see `explainer.ipynb` and `CODEBOOK.md`.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 160)
pd.set_option('display.max_rows', 100)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

texts = pd.read_csv('data/religious_texts.csv', low_memory=False)      # every verse, rule or paragraph
tagged = pd.read_csv('data/rules_tagged.csv', low_memory=False)        # keyword tags, including topic
rules = pd.read_csv('data/coded_rules.csv', low_memory=False)          # the hand-coded rules
agreement = pd.read_csv('data/coding/agreement.csv', dtype=str)        # both coders on 200 rules

# add each rule's keyword-based topic (marriage, purity, dress...)
rules = rules.merge(tagged[['id', 'domain']], on='id', how='left')

# the Bible is easier to compare split by Testament
rules['text_group'] = rules.section.fillna(rules.religion).where(rules.religion != 'Christianity',
                                                                 'Christianity: ' + rules.section.fillna(''))
print(f'{len(texts):,} text rows, {len(rules):,} coded rules')

In [ ]:
GOVERNS = {'M': 'men', 'W': 'women', 'B': 'both', 'G': 'everyone'}
EFFECT = {'R': 'restricts', 'P': 'protects', 'X': 'mixed', 'N': 'none'}

def decode(code):
    """Turn a code such as '1MPR' into words."""
    if code == '0':
        return 'not a rule'
    return f'governs {GOVERNS[code[1]]}; women: {EFFECT[code[2]]}; men: {EFFECT[code[3]]}'

def search(word, table=rules, religion=None, n=20):
    """Find rows whose text contains a word or regular expression."""
    hits = table[table.text.str.contains(word, case=False, regex=True, na=False)]
    if religion:
        hits = hits[hits.religion == religion]
    print(f'{len(hits)} matches for {word!r}')
    columns = [c for c in ['religion', 'book', 'chapter', 'verse', 'code', 'text'] if c in hits]
    return hits[columns].head(n)

def bar(series, title=''):
    """Quick horizontal bar chart of a pandas Series."""
    ax = series.sort_values().plot.barh(figsize=(8, max(2.5, len(series) * 0.42)), color='#1d4ed8', width=0.7)
    ax.set_title(title, loc='left')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

decode('1MPR')

## Experiment 1: search for a word

Change `word` to anything: `veil`, `inherit`, `widow`, `divorce`. It searches the coded rules. Pass `table=texts` to search every verse instead.

In [ ]:
word = 'veil'
search(word)

## Experiment 2: filter by code

Each letter of `code` can be tested separately. This example finds women-only rules that restrict women in one religion.

In [ ]:
religion = 'Islam'
picked = rules[(rules.religion == religion) & (rules.code.str[1] == 'W') & (rules.code.str[2] == 'R')]
picked[['book', 'chapter', 'verse', 'code', 'text']]

## Experiment 3: topics

Which areas of life do women-only and men-only rules cover? `domain` comes from the keyword tagger, so treat it as approximate. Change `who` to `'M'`, `'W'`, `'B'` or `'G'`.

In [ ]:
who = 'W'
topics = pd.crosstab(rules[rules.code.str[1] == who].religion, rules[rules.code.str[1] == who].domain)
topics

## Experiment 4: your own measure

Here, the share of one-gender rules that apply to women. Try changing the formula or the rows included.

In [ ]:
governs = rules.code.str[1]
counts = pd.crosstab(rules.text_group, governs)
women_share = counts['W'] / (counts['W'] + counts['M'])
bar(women_share, 'Women-only rules as a share of all one-gender rules')
women_share.round(2)

## Experiment 5: test a judgement call

The two coders disagreed about whether "he" in Manusmriti means men only or everyone. This recounts Manusmriti's men-only rules assuming a share of them are really general. Change `share_generic` to see how much the conclusion moves.

In [ ]:
share_generic = 0.25   # try values between 0 and 0.6

manu = rules[rules.religion == 'Hinduism']
men = (manu.code.str[1] == 'M').sum()
women = (manu.code.str[1] == 'W').sum()
adjusted_men = round(men * (1 - share_generic))
print(f'Manusmriti: {men} men-only as coded; {adjusted_men} if {share_generic:.0%} are general; women-only stays {women}')

## Experiment 6: where the coders disagreed

In [ ]:
disagreed = agreement[agreement.coder1 != agreement.coder2]
print(f'{len(disagreed)} of {len(agreement)} rules coded differently')
disagreed[['religion', 'coder1', 'coder2', 'text']].head(15)

## Your experiments

Add your own work below. `texts`, `tagged`, `rules` and `agreement` are already loaded.